In [1]:
import sys, os, shutil
import h5py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append('/home/nexus-admin/NEXUS_RF/BackendTools')
import TimestreamHelperFunctions as Thf
import PyMKID_USRP_functions as PUf
import PyMKID_resolution_functions as Prf
import OptFilterTools as oft

In [2]:
target_V = 4.00 
out_path = "/home/nexus-admin/Downloads/BetaData"

all_series = np.array([
    "20230215_132909",
    "20230215_131026",
    "20230215_125136",
])

In [3]:
datapath = '/data/USRP_Laser_TempScan_Data'
series   = all_series[2]

sum_file, dly_file, vna_file, nse_files, led_files = Thf.GetFiles(series, 
                                                        base_path=datapath,
                                                        sep_noise_laser=True,
                                                        verbose=False)

voltages, p_params, charFs, charZs = oft.parse_metadata(sum_file, 
                                                        blank_fraction=0.1, 
                                                        verbose=False)

n_runs   = len(led_files)
for f in nse_files:
    print(f)
    
print(p_params['rf_power'])

OSError: Unable to open file (unable to open file: name = 'None', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [ ]:
powers, PSDs, res, timestreams = Thf.CleanPSDs(nse_files[0], vna_file, 
        PSD_lo_f=1e1, 
        PSD_hi_f=5e4, 
        rem_dec=10,
        f_transient=p_params["blank_fraction"], 
        charFs=charFs[0].real, 
        charZs=charZs[0])

print(timestreams['radius coefficient'])
print(timestreams['arc coefficient'])

In [ ]:
LED_files =  np.array([ led_files[np.argmin(np.abs(voltages-target_V))] ])
print(LED_files)

In [ ]:
mean_dict, sdev_dict, maxv_dict = oft.plot_pulse_windows(LED_files, nse_files[0], vna_file, p_params, 
    p1=5, p2=90, decimate_down_to=None, pulse_cln_dec=4,
    PHASE=True, show_plots=True)

In [ ]:
cut_df = oft.define_default_cuts(LED_files, mean_dict, sdev_dict, maxv_dict, PHASE=True, p1=5, p2=90, force_save=True)

In [ ]:
_i = 0
# cut_df = oft.update_cut_limit(cut_df, LED_files, _i, "mean_min", np.nan)
# cut_df = oft.update_cut_limit(cut_df, LED_files, _i, "mean_max", np.nan)
# cut_df = oft.update_cut_limit(cut_df, LED_files, _i, "sdev_min", np.nan)
# cut_df = oft.update_cut_limit(cut_df, LED_files, _i, "sdev_max", np.nan)
# cut_df = oft.update_cut_limit(cut_df, LED_files, _i, "wfmx_min", np.nan)
cut_df = oft.update_cut_limit(cut_df, LED_files, _i, "wfmx_max", 0.70)

if False:
    oft.save_cut_df(cut_df, LED_files, PHASE=PHASE)

cut_df

In [ ]:
bad_pls_idxs = oft.get_bad_pulse_idxs(LED_files, cut_df, mean_dict, sdev_dict, maxv_dict)

In [ ]:
# oft.clean_pulse_windows(LED_files, nse_files[0], vna_file, p_params, bad_pls_idxs, decimate_down_to=None, pulse_cln_dec=4,PHASE=True)

In [ ]:
show_plots = True

## Options
verbose = False 

## Window selection for pulse-free region
window_shift_seconds = 0 # -8.0e-3

noise_file = nse_files[0]

decimate_down_to = 5e4
pulse_cln_dec = 4 #None
PHASE=True

j = 0
for pulse_file in LED_files:
    print('===================')
    print('cleaning pulse file:',pulse_file)
    print('using VNA file:     ',vna_file)
    print('using noise file:   ',noise_file)

    ## Get the decimated timestream and frequency step
    pulse_noise, N, T, t, f, samp_rate = oft.get_decimated_timestream(pulse_file, p_params, decimate_down_to, pulse_cln_dec)
    time = 1e3*(p_params["time_btw_pulse"]-t[::-1])
    pulse_noise, pulse_info = PUf.unavg_noi(pulse_file)

    ## Define the regions where pulses exist
    ## =====================================

    ## This defines where (in # of pulse windows) to start looking for pulse windows
    pulse_start = int(p_params["total_pulses"] * p_params["blank_fraction"])
    samples_per_pulse = int(p_params["time_btw_pulse"]*samp_rate)
    if verbose:
        print("Starting pulse partitioning after", pulse_start, "windows (of",p_params["total_pulses"],")")

    ## How many samples to shift the pulse window definition
    window_shift = int(window_shift_seconds * samp_rate)
    if verbose:
        print("Shifting pulse window by", window_shift, "samples")

    ## Create empty arrays to store our results in
    noise_averages = np.zeros((3),dtype=np.complex128)
    J_r = np.zeros((N,3)); J_arc = np.zeros((N,3))

    ## Create empty arrays to store values for histograms
    bl_means = np.array([],dtype=np.complex128)
    bl_sdevs = np.array([])#,dtype=np.complex128)

    ## Create a plot to store waveforms
    if show_plots:
        fi0 = plt.figure(pulse_file+"_a")
        ax0 = fi0.gca()
        ax0.set_xlabel("Time [ms]")
        ax0.set_ylabel(r"$\log_{10}|S_{21}|$")
        ax0.set_title(".".join(pulse_file.split("/")[-1].split(".")[0:-1]))

        fi1 = plt.figure(pulse_file+"_b")
        ax1 = fi1.gca()
        ax1.set_xlabel(r"$\Re(S_{21})$")
        ax1.set_ylabel(r"$\Im(S_{21})$")
        ax1.set_title(".".join(pulse_file.split("/")[-1].split(".")[0:-1]))

    ## Count how many good pulses there are
    n_good_pulses = p_params["num_pulses"] - len(bad_pls_idxs[pulse_file])

    ## Start the loop over pulse windows
    k=0
    for pulse_i in range(pulse_start,int(p_params["total_pulses"]),1):

        ## Skip the bad pulse windows
        if k in bad_pls_idxs[pulse_file]:
            ## Increment the counter
            k += 1
            continue

        ## Define the sample index where this pulse window ends
        pulse_i_end = int((pulse_i+1)*samples_per_pulse)

        ## Define the start of the pulse free region (period after pulse, before the next one, where it should be baseline noise)
        no_pulse_idx_start = pulse_i_end + window_shift - N 

        ## Define the end of the window (where the pulse-free region ends)
        no_pulse_idx_end   = pulse_i_end + window_shift

        ## Grab the timestream in that region and average it
        no_pulse_chunk = pulse_noise[no_pulse_idx_start:no_pulse_idx_end,:]

        ## Calculate some means and stdevs of this pulse-free timestream
        m = np.mean(no_pulse_chunk,axis=0,dtype=np.complex128) ; bl_means = np.append(bl_means,m[0])

        if PHASE:
            s = np.std( np.angle(    no_pulse_chunk[:,0])  ) ; bl_sdevs = np.append(bl_sdevs,s)
        else:
            s = np.std( np.log10(abs(no_pulse_chunk[:,0])) ) ; bl_sdevs = np.append(bl_sdevs,s)

        ## Keep a running average of the noise across all pulse regions
        noise_averages += m / n_good_pulses    

        ## Plot the pulse free region against time
        if show_plots: # and (k==0):
            ax0.plot(time, np.log10(abs(no_pulse_chunk[:,0])),alpha=0.25)
            ax1.scatter(no_pulse_chunk[:,0].real,no_pulse_chunk[:,0].imag,alpha=0.25)

        ## Convert to the electronics basis and compute the J objects
        r_chunk,arc_chunk,_,_= Prf.electronics_basis(no_pulse_chunk)
        J_r += abs(Prf.discrete_FT(r_chunk))**2 / n_good_pulses * 2 * T
        J_arc += abs(Prf.discrete_FT(arc_chunk))**2 / n_good_pulses * 2 * T

        ## Increment the counter
        k += 1

    if verbose:
        print("Searched",n_good_pulses,"pulse windows")
        print('used ' + str(n_good_pulses) + ' chunks to find quiescent point')

    if show_plots:
        ax0.axhline(y=np.log10(abs(noise_averages[0])),color="k",ls='--')

        fi2 = plt.figure(pulse_file+"_c")
        ax2 = fi2.gca()
        if PHASE:
            ax2.hist(np.angle(bl_means))
            ax2.set_xlabel(r"Pre-trigger BL mean $\arg(S_{21})$")
        else:
            ax2.hist(np.log10(abs(bl_means)))
            ax2.set_xlabel(r"Pre-trigger BL mean $\log_{10}(|S_{21}|)$")
        ax2.set_ylabel("Occurences")
        ax2.set_title(".".join(pulse_file.split("/")[-1].split(".")[0:-1]))

        fi3 = plt.figure(pulse_file+"_d")
        ax3 = fi3.gca()
        if PHASE:
            ax3.hist(bl_sdevs)
            ax3.set_xlabel(r"Pre-trigger BL sdev $\arg(S_{21})$")
        else:
            ax3.hist(bl_sdevs)
            ax3.set_xlabel(r"Pre-trigger BL sdev $\log_{10}(|S_{21}|)$")
        ax3.set_ylabel("Occurences")
        ax3.set_title(".".join(pulse_file.split("/")[-1].split(".")[0:-1]))

    ## Pull the two real quantities from the complex timestream averages
    radius_averages = abs(noise_averages)
    angle_averages  = np.angle(noise_averages)
    if verbose:
        print(radius_averages)
        print(angle_averages)

    ## Rotate the timestream by the averange angle, then get the rotated phase timestream
    pulse_timestream_rotated = pulse_noise*np.exp(-1j*angle_averages)
    angle_timestream = np.angle(pulse_timestream_rotated)

    ## Subtract off the average magnitude value and calculate an arc length
    radius = abs(pulse_noise) - radius_averages
    arc    = angle_timestream*radius_averages

    ## Create output containers for the clean timestreams
    radius_clean = np.zeros(radius.shape)
    arc_clean    = np.zeros(arc.shape)

    if verbose:
        print('built radius and arc length timestreams given by quiescent point')
        print(noise_file)

    ## Pull the dictionary containing cleaning coefficients from the noise timestream
    _,data_info = PUf.clean_noi(noise_file[:-3]+'_cleaned.h5')

    ## Loop over each tone in the radius timestream
    for t in range(radius.shape[1]):
        ## Pull the coefficients from the noise cleaning
        radius_coefficient = data_info['radius cleaning coefficient'][t]
        arc_coefficient    = data_info['arc cleaning coefficient'][t]

        ## Clean each tone with the off-resonance tones
        if t == 0:
            off_tone_idcs = [1,2]
        elif t == 1:
            off_tone_idcs = [2]
        elif t == 2:
            off_tone_idcs = [1]

        ## Perform the radius cleaning
        off_tone_radius = np.mean(radius[:,off_tone_idcs],axis=1,dtype=np.float64)
        radius_clean[:,t] = radius[:,t] - radius_coefficient*off_tone_radius

        ## Perform the arc length cleaning
        off_tone_arc = np.mean(arc[:,off_tone_idcs],axis=1,dtype=np.float64)
        arc_clean[:,t] = arc[:,t] - arc_coefficient*off_tone_arc

        if verbose: 
            print('cleaned tone ' + str(t))

    ## Subtract off the mean from cleaned radius and arc length timestreams
    radius_clean -= np.mean(radius_clean,axis=0,dtype='float64')
    arc_clean -= np.mean(arc_clean,axis=0,dtype='float64')

    ## Save the clean timestreams to a file
    pulse_noise_clean = Prf.save_clean_timestreams(pulse_file,
                                                   radius_averages,
                                                   angle_averages,
                                                   radius_clean,
                                                   arc_clean,
                                                   samp_rate,
                                                   timestreams['radius coefficient'],
                                                   timestreams['arc coefficient'],
                                                   override=True)

    ## Calculate the PSDs for each of the cleaned pulses

    ## Create containers for our output PSDs
    J_r_clean = np.zeros((N,3)); J_arc_clean = np.zeros((N,3))

    ## Loop over pulses
    k = 0
    for pulse_i in range(pulse_start,int(p_params["total_pulses"]),1):
        ## Skip the bad pulse windows
        if k in bad_pls_idxs[pulse_file]:
            ## Increment the counter
            k += 1
            continue

        ## Define the sample index where this pulse window ends
        pulse_i_end = int((pulse_i+1)*samples_per_pulse) 

        ## Define the start of the pulse free region (period after pulse, before the next one, where it should be baseline noise)
        no_pulse_idx_start = pulse_i_end + window_shift - N 

        ## Define the end of the window (where the pulse-free region ends)
        no_pulse_idx_end   = pulse_i_end + window_shift

        ## Grab the timestream in that region
        no_pulse_chunk = pulse_noise_clean[no_pulse_idx_start:no_pulse_idx_end,:]

        ## Convert the pulse-free region to electronics basis
        r_chunk,arc_chunk,_,_= Prf.electronics_basis(no_pulse_chunk)

        ## Compute the PSDs
        J_r_clean += abs(Prf.discrete_FT(r_chunk))**2 / n_good_pulses * 2 * T
        J_arc_clean += abs(Prf.discrete_FT(arc_chunk))**2 / n_good_pulses * 2 * T

        ## Increment the counter
        k += 1

    ## Trim the output to the positive frequency region only
    J_r = J_r[f>=0]; J_r_clean = J_r_clean[f>=0]; J_arc = J_arc[f>=0]; J_arc_clean = J_arc_clean[f>=0]

    ## Every tenth files, show the PSDs
    if j % 1 == 0:
        print(pulse_file)
        fig_0, axes_0 = plt.subplots(2,3,sharex=True,sharey='row',figsize=(5*3,10))

        Prf.plot_PSDs(f[f>0],J_r,J_arc,pulse_file,\
                      ['radius','arc length'],units=['ADCu','ADCu'],savefig='electronics',\
                      data_freqs=pulse_info['search freqs'],\
                      P_1_clean=J_r_clean,P_2_clean=J_arc_clean,\
                      fig_0=fig_0,axes_0=axes_0)

    j += 1

In [ ]:
fraction_to_keep = 0.5 # 1.0 # 0.5 # (5.0/12.5) # 1.0
window_shift_seconds = 0 # 5.0e-3
verbose = False

j = 0 
for pulse_file in LED_files:
    
    ## Get the VNA data for this set of runs
    f,z = PUf.read_vna(vna_file)

    print('===================')
    print('averaging pulse file: ' + pulse_file)

    ## Load the clean pulse data
    clean_pulse_file = pulse_file[:-3] + '_cleaned.h5'
    pulse_noise_clean,data_info = PUf.clean_noi(clean_pulse_file)
    if verbose: 
        print('loaded clean pulse data')        
        print('sampling_rate: ' + str(data_info['sampling_rate']))
    
    ## Determine how many samples are in each pulse window
    samples_per_pulse = data_info['sampling_rate'] * p_params["time_btw_pulse"]

    ## Do extra decimation if needed (1 = no decimation) 
    decimation = 1
    time = Prf.average_decimate(pulse_info['time'],decimation)
    pulse_noise_clean = Prf.average_decimate(pulse_noise_clean,decimation)
    
    ## Update the samples per window and sampling rate with new decimation
    samples_per_pulse_decimated = int(samples_per_pulse / decimation)
    sampling_rate = data_info['sampling_rate'] / decimation
    if verbose:
        print('further decimation by ' + str(decimation) + ' complete')

    ## Create a container to store our average pulse in complex S21 for this file
    pulse_avg    = np.zeros(int(samples_per_pulse_decimated*fraction_to_keep),dtype=np.complex128)
    
    ## Determine how many samples to shift the window by
    window_shift = int(window_shift_seconds*sampling_rate)
    
    ## Identify the first pulse window after the transient period
    pulse_start  = int(p_params["total_pulses"] * p_params["blank_fraction"])
    
    ## Count how many good pulses there are in this file
    n_good_pulses = p_params["num_pulses"] - len(bad_pls_idxs[pulse_file])
    
    ## Start the loop over pulse windows
    k=0
    for pulse_i in range(pulse_start,int(p_params["total_pulses"]),1):
        
        ## Skip the bad pulse windows
        if k in bad_pls_idxs[pulse_file]:
            ## Increment the counter
            k += 1
            continue

        ## Define the sample index where this pulse window ends
        pulse_idx_start = int((pulse_i  )*samples_per_pulse_decimated) + window_shift
        # pulse_idx_end   = int((pulse_i+1)*samples_per_pulse_decimated) + window_shift -1
        pulse_idx_end   = int(round((pulse_i+fraction_to_keep)*samples_per_pulse_decimated,0)) + window_shift
        
        ## Create a list of indeces corresponding to the samples in this pulse window
        pulse_idx_list = np.arange(pulse_idx_start,pulse_idx_end,1,dtype=int)
        
        ## Average the pulses in each good window
        pulse_avg += pulse_noise_clean[pulse_idx_list,0] / n_good_pulses
        
        ## Increment the counter
        k += 1
    
    ## Create a figure in complex S21 to show VNA, full timestream, and average pulse
    fig, axs = plt.subplots(1, 2)
    plt.suptitle(pulse_file.split("/")[-1])
    axs[0].set_xlabel(r"$\Re(S_{21})$")
    axs[0].set_ylabel(r"$\Im(S_{21})$")
    axs[1].set_xlabel("Time [ms]")
    if PHASE:
        axs[1].set_ylabel(r"$\arg (S_{21})$")
    else:
        axs[1].set_ylabel(r"$\log10 |S_{21}|$")
    
    axs[0].plot(pulse_noise_clean[:,0].real,pulse_noise_clean[:,0].imag,ls='',marker='.',alpha=0.1,color='grey')
    axs[0].plot(pulse_avg.real,pulse_avg.imag,color='C'+str(j % 10),ls='-',marker='.')
    axs[0].plot(z.real,z.imag,color='k',ls='-',marker='.',alpha=1.00)
    
    if PHASE:
        axs[1].plot(time[pulse_idx_list,0]*1e3,np.angle(pulse_noise_clean[pulse_idx_list,0]))
        axs[1].plot(time[pulse_idx_list,0]*1e3,np.angle(pulse_avg))
    else:
        axs[1].plot(time[pulse_idx_list,0]*1e3,np.log10(abs(pulse_noise_clean[pulse_idx_list,0])))
        axs[1].plot(time[pulse_idx_list,0]*1e3,np.log10(abs(pulse_avg)))
        
    pulse_average_time = time[pulse_idx_list,0]*1e3
    pulse_average_cplx = pulse_avg
    
    width = 50 * np.std(pulse_noise_clean[:,0].real)
    x_c = np.mean(pulse_avg.real)
    y_c = np.mean(pulse_avg.imag)
    axs[0].set_xlim([x_c - width/2., x_c + width/2.])
    axs[0].set_ylim([y_c - width/2., y_c + width/2.])
    axs[0].set_aspect('equal','box')
    
    plt.subplots_adjust(wspace=0.5)
    
    # plt.savefig('noise and averaged pulse.png',dpi=100)
        
    print('Used ' + str(n_good_pulses) + ' pulses to average')
    with h5py.File(clean_pulse_file, "a") as fyle:
        if 'pulse_shape' in fyle.keys():
            del fyle['pulse_shape']
            print('deleted an old pulse shape')
        fyle.create_dataset('pulse_shape',data = np.asarray(pulse_avg))
    j += 1

In [ ]:
with h5py.File(clean_pulse_file, "r") as fyle:
    plt.plot(pulse_average_time,np.angle(fyle['pulse_shape']))

In [ ]:
plt.plot(pulse_average_time,np.angle(pulse_average_cplx))

In [ ]:
plt.plot(pulse_average_cplx.real,pulse_average_cplx.imag)

In [ ]:
## Copy the VNA file
shutil.copy(sum_file,    os.path.join(out_path,sum_file.split("/")[-1]))

## Copy the VNA file
shutil.copy(vna_file,    os.path.join(out_path,vna_file.split("/")[-1]))

## Copy the Noise file
shutil.copy(nse_files[0],os.path.join(out_path,nse_files[0].split("/")[-1]))

## Copy the Cleaned Pulse file
shutil.copy(clean_pulse_file,os.path.join(out_path,clean_pulse_file.split("/")[-1]))